# 01 · Data profiling

Displays the outputs of `python -m ssn data validate` and `python -m ssn data profile`.
No computation is duplicated here: every table is read from `reports/tables/`. Re-run the CLI to refresh.

Dataset: UCI 697, *Predict Students' Dropout and Academic Success* (CC BY 4.0). See `data/README.md`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from ssn.paths import repo_root

ROOT = repo_root()
T = ROOT / 'reports' / 'tables'
assert (ROOT / 'data' / 'raw' / 'data.csv').is_file(), 'run: python -m ssn data download'
report = json.loads((T / 'validate_report.json').read_text())
print('rows x cols:', report['n_rows'], 'x', report['n_cols'])
print('schema errors:', report['errors'])
report['classification_counts']

## Target distribution and derived `is_dropout`

In [ ]:
target = pd.read_csv(T / 'profile_target.csv')
print('is_dropout positive rate:', round(target.loc[target.is_dropout == 1, 'count'].sum() / target['count'].sum(), 4))
target

## Per-column profile (missingness, uniqueness, ranges)

In [ ]:
cols = pd.read_csv(T / 'profile_columns.csv')
print('total missing cells:', int(cols.n_missing.sum()))
cols[['column', 'availability', 'role', 'dtype', 'n_missing', 'n_unique', 'min', 'p50', 'max']]

## Duplicates and invalid values

In [ ]:
display(pd.read_csv(T / 'profile_duplicates.csv'))
inv = pd.read_csv(T / 'profile_invalid_values.csv')
inv[inv['count'] > 0] if (inv['count'] > 0).any() else print('no documented-code or range violations')

## Sensitive attributes (aggregate counts only; audit-only, never model inputs)

In [ ]:
cv = pd.read_csv(T / 'profile_categorical_values.csv')
cv[cv.column.isin(['Gender', 'International', 'Educational special needs', 'Marital status', 'Nacionality'])].sort_values(['column', 'count'], ascending=[True, False])

## Outlier candidates (IQR fences). Counts only; treatment is decided and justified in Milestone 3.

In [ ]:
pd.read_csv(T / 'profile_outliers.csv')

## Preview of de-identified allow-listed columns (first 5 rows)

Only allow-listed model inputs are shown; sensitive columns, second-semester columns, and the outcome are omitted.

In [ ]:
from ssn.data.schema import load_raw
from ssn.features import allowlist as al

allow = al.load(ROOT / 'configs' / 'features.yaml')
df = load_raw(ROOT / 'data' / 'raw' / 'data.csv')
allow.project_features(df).head()